In [1]:
# ── Cell 1: Colab CPU Setup ──────────────────────────────────────────────────
# No GPU required for this notebook (API calls only)

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

os.makedirs(f'{PROJECT_DIR}/data/augmented', exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai', 'anthropic', 'tqdm', 'pandas', 'numpy'
], check=True)

sys.path.insert(0, f'{PROJECT_DIR}/src')
print(f'PROJECT_DIR = {PROJECT_DIR}')


Running locally — skipping Drive mount
PROJECT_DIR = /Users/aman2394/Desktop/ICWSM-2026-FMD


In [2]:
# ── Cell 2: Load Raw Data, Show Stats ────────────────────────────────────────
# RFC-BENCH schema:
#   "Open-ended Verifiable Question"  →  prompt prefix + paragraph text
#   "Ground-True Answer"              →  "The provided information is true/false."

import pandas as pd
from utils.data_loader import load_combined_data, records_to_texts_labels
import textwrap

records = load_combined_data(PROJECT_DIR)
texts, labels = records_to_texts_labels(records)

df = pd.DataFrame(records)
print(df[['id', 'split', 'label', 'text']].head())
print()
print('Label distribution by split:')
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))
print()
df['text_len'] = df['text'].str.len()
print('Text length stats (chars):')
print(df['text_len'].describe().round(1))
print()
print('=== Sample records ===')
for lbl in [1, 0]:
    row = df[df['label'] == lbl].iloc[0]
    print(f"[label={lbl}] ({row['split']})")
    print(textwrap.fill(row['text'], width=100))


SFT: 1000 samples  (true=500, false=500)
RL:  1000 samples  (true=500, false=500)

Total: 2000 samples  (true=1000, false=1000, balance=50.0% true)
   id split  label                                               text
0   0   sft      0  NVIDIA’s Way Ahead Of Broadcom (AVGO), Says Ji...
1   1   sft      0  Former Goldman Sachs CEO during the 2008 crash...
2   2   sft      1  Applied Digital (APLD) Stock Trades Up, Here I...
3   3   sft      1  Zoetis CFO talks Q2 beat & education's role in...
4   4   sft      1  NIO, XPeng, Li Auto Log Strong August Deliveri...

Label distribution by split:
label    0    1
split          
rl     500  500
sft    500  500

Text length stats (chars):
count    2000.0
mean      373.1
std       174.3
min       115.0
25%       219.0
50%       335.0
75%       519.0
max      1675.0
Name: text_len, dtype: float64

=== Sample records ===
[label=1] (sft)
Applied Digital (APLD) Stock Trades Up, Here Is Why Shares of digital infrastructure provider
Applied Digital (

In [ ]:
# ── Cell 3: Run Augmentation ──────────────────────────────────────────────────
import os

os.environ['LLM_PROVIDER']              = 'azure'
os.environ['AZURE_OPENAI_API_KEY']      = '<YOUR_AZURE_API_KEY>'
os.environ['AZURE_OPENAI_ENDPOINT']     = '<YOUR_AZURE_ENDPOINT>'
os.environ['AZURE_OPENAI_DEPLOYMENT']   = '<YOUR_DEPLOYMENT_NAME>'
os.environ['AZURE_OPENAI_API_VERSION']  = '<YOUR_API_VERSION>'

from augmentation.call_llm import augment_dataset

OUTPUT_PATH = f'{PROJECT_DIR}/data/augmented/augmented_train.json'

augment_dataset(
    project_dir=PROJECT_DIR,
    output_path=OUTPUT_PATH,
    perturbations_per_sample=3,
    delay=0.5,
    resume=True,
)


SFT: 1000 samples  (true=500, false=500)
RL:  1000 samples  (true=500, false=500)

Total: 2000 samples  (true=1000, false=1000, balance=50.0% true)
Found 1000 True samples to augment.


Augmenting:   0%|          | 0/1000 [00:00<?, ?it/s]

⚠️  Error on sample 2 / false_attribution: Error code: 401 - {'error': {'message': 'Incorrect API key provided: beb355a4********************0257. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
⚠️  Error on sample 2 / cherry_picked_context: Error code: 401 - {'error': {'message': 'Incorrect API key provided: beb355a4********************0257. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
⚠️  Error on sample 2 / misleading_framing: Error code: 401 - {'error': {'message': 'Incorrect API key provided: beb355a4********************0257. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


Augmenting:   0%|          | 1/1000 [00:16<4:34:55, 16.51s/it]

⚠️  Error on sample 3 / hedge_manipulation: Error code: 401 - {'error': {'message': 'Incorrect API key provided: beb355a4********************0257. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


Augmenting:   0%|          | 1/1000 [00:17<4:48:54, 17.35s/it]


KeyboardInterrupt: 

In [ ]:
# ── Cell 4: Validate Output ───────────────────────────────────────────────────
import json, pandas as pd

aug_path = f'{PROJECT_DIR}/data/augmented/augmented_train.json'
with open(aug_path) as f:
    augmented = json.load(f)

df_aug = pd.DataFrame(augmented)
print(f'Total records : {len(df_aug)}')
print()
print('Label distribution:')
print(df_aug['label'].value_counts())
print()
if 'perturbation_type' in df_aug.columns:
    print('Records per perturbation_type (None = original):')
    print(df_aug['perturbation_type'].value_counts(dropna=False))
    print()

df_aug['text_len'] = df_aug['text'].str.len()
print('Text length by label:')
print(df_aug.groupby('label')['text_len'].describe().round(1))
print()

sample = df_aug[df_aug['label'] == 0].iloc[0]
print('Sample augmented False record:')
print(f'  perturbation_type : {sample.get("perturbation_type")}')
print(f'  text              : {sample["text"][:300]}')
print()
print('✅ Data ready for feature extraction (notebook 02).')
